# Download Latest SFT Blob Run

This notebook downloads the most recent SFT training run stored under `runs/sft/<timestamp>/` in Azure Blob Storage and saves it locally into `aca_sft/model_run/`.

It expects these environment variables to be available in `.env`:

- `AZURE_STORAGE_CONNECTION_STRING`
- `AZURE_BLOB_CONTAINER`


In [1]:
from pathlib import Path

from dotenv import load_dotenv

CWD = Path.cwd().resolve()
SEARCH_ROOTS = [CWD, *CWD.parents]
ENV_CANDIDATES = []

for root in SEARCH_ROOTS:
    ENV_CANDIDATES.append(root / '.env')
    ENV_CANDIDATES.append(root / 'aca_sft' / '.env')

seen = set()
ENV_CANDIDATES = [path for path in ENV_CANDIDATES if not (str(path) in seen or seen.add(str(path)))]

for env_path in ENV_CANDIDATES:
    if env_path.exists():
        load_dotenv(env_path, override=False)
        NOTEBOOK_DIR = env_path.parent if env_path.parent.name == 'aca_sft' else CWD
        print(f'Loaded environment from: {env_path}')
        break
else:
    NOTEBOOK_DIR = CWD
    print('No .env file found in the working directory tree; relying on existing environment variables.')

print(f'Working directory: {CWD}')
print(f'Notebook asset directory: {NOTEBOOK_DIR}')

Loaded environment from: C:\Users\deril\OneDrive\Desktop\Deril\Development\Transformers-From-Scratch\aca_sft\.env
Working directory: C:\Users\deril\OneDrive\Desktop\Deril\Development\Transformers-From-Scratch
Notebook asset directory: C:\Users\deril\OneDrive\Desktop\Deril\Development\Transformers-From-Scratch\aca_sft


In [2]:
import os
from collections import defaultdict

from azure.storage.blob import BlobServiceClient

CONNECTION_STRING = os.environ['AZURE_STORAGE_CONNECTION_STRING']
CONTAINER_NAME = os.environ['AZURE_BLOB_CONTAINER']
RUNS_PREFIX = 'runs/sft/'
DOWNLOAD_DIR = NOTEBOOK_DIR / 'model_run'
INCLUDE_CHECKPOINTS = False

blob_service = BlobServiceClient.from_connection_string(CONNECTION_STRING)
container_client = blob_service.get_container_client(CONTAINER_NAME)

print(f'Container: {CONTAINER_NAME}')
print(f'Download directory: {DOWNLOAD_DIR}')
print(f'Include checkpoints: {INCLUDE_CHECKPOINTS}')

Container: gpt-scratch
Download directory: C:\Users\deril\OneDrive\Desktop\Deril\Development\Transformers-From-Scratch\aca_sft\model_run
Include checkpoints: False


In [3]:
blobs = list(container_client.list_blobs(name_starts_with=RUNS_PREFIX))

if not blobs:
    raise RuntimeError("No blobs found under the 'runs/sft/' prefix.")

run_to_blobs = defaultdict(list)

for blob in blobs:
    parts = blob.name.split('/', 3)
    if len(parts) < 3:
        continue
    run_id = parts[2]
    run_to_blobs[run_id].append(blob)

if not run_to_blobs:
    raise RuntimeError("No SFT run folders were detected under the 'runs/sft/' prefix.")

latest_run_id = sorted(run_to_blobs.keys())[-1]
latest_run_blobs = sorted(run_to_blobs[latest_run_id], key=lambda blob: blob.name)
if not INCLUDE_CHECKPOINTS:
    skipped_checkpoint_blobs = [blob for blob in latest_run_blobs if '/checkpoints/' in blob.name]
    latest_run_blobs = [blob for blob in latest_run_blobs if '/checkpoints/' not in blob.name]
else:
    skipped_checkpoint_blobs = []

print(f'Latest SFT run detected: {latest_run_id}')
print(f'Blob count in latest SFT run: {len(latest_run_blobs)}')
print(f'Skipped checkpoint blobs: {len(skipped_checkpoint_blobs)}')

for blob in latest_run_blobs:
    print(blob.name)

Latest SFT run detected: 2026-03-27T14-05-30Z
Blob count in latest SFT run: 3
Skipped checkpoint blobs: 2
runs/sft/2026-03-27T14-05-30Z/logs/train.log
runs/sft/2026-03-27T14-05-30Z/metrics/metrics.json
runs/sft/2026-03-27T14-05-30Z/model/sft_final.pth


In [4]:
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

downloaded_files = []

for blob in latest_run_blobs:
    relative_path = blob.name.removeprefix(f'runs/sft/{latest_run_id}/')
    if not relative_path:
        continue

    local_path = DOWNLOAD_DIR / relative_path
    local_path.parent.mkdir(parents=True, exist_ok=True)

    with open(local_path, 'wb') as file_obj:
        stream = container_client.download_blob(blob.name)
        file_obj.write(stream.readall())

    downloaded_files.append(local_path)
    print(f'Downloaded: {blob.name} -> {local_path}')

print(f'\nDownloaded {len(downloaded_files)} files into {DOWNLOAD_DIR}')

Downloaded: runs/sft/2026-03-27T14-05-30Z/logs/train.log -> C:\Users\deril\OneDrive\Desktop\Deril\Development\Transformers-From-Scratch\aca_sft\model_run\logs\train.log
Downloaded: runs/sft/2026-03-27T14-05-30Z/metrics/metrics.json -> C:\Users\deril\OneDrive\Desktop\Deril\Development\Transformers-From-Scratch\aca_sft\model_run\metrics\metrics.json
Downloaded: runs/sft/2026-03-27T14-05-30Z/model/sft_final.pth -> C:\Users\deril\OneDrive\Desktop\Deril\Development\Transformers-From-Scratch\aca_sft\model_run\model\sft_final.pth

Downloaded 3 files into C:\Users\deril\OneDrive\Desktop\Deril\Development\Transformers-From-Scratch\aca_sft\model_run


In [5]:
for path in sorted(DOWNLOAD_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(DOWNLOAD_DIR))

logs\train.log
metrics\metrics.json
model\sft_final.pth
